<a href="https://colab.research.google.com/github/fmssilva/DL_Proj/blob/main/assignment_1/task2/task2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pokemon Type Classification | DL Assignment 1

## Task 2 — CNN

**Goal:** classify 3600 Pokemon images into 9 types using CNNs. Compare against MLP baseline from Task 1.

**Metric:** Macro-averaged F1 — used because class imbalance is 2.76× (Water 674 vs Ground 244).

**Budget:** 1 hour GPU on Google Colab free tier.

**Strategy — progressive refinement, one variable at a time:**

| Step | What | Why |
|------|------|-----|
| §2.1 | Architecture sweep on clean data | Find the best CNN backbone |
| §2.2 | Architecture refinement | Fine-tune the top backbone(s) |
| §2.3 | Augmentation on the winners | Improve generalisation via data diversity |
| §2.4 | Regularisation & loss tuning | Fine-tune the training recipe |
| §2.5 | Ablations on the final model | Confirm each component earns its place |
| §3   | Ensembles | Combine complementary models for extra F1 |

Each step picks its starting point from `get_best_models()`, so winners flow automatically.

_____________
**DO NOT RUN ALL CELLS**

> This notebook requires ~30 min to run all training sequences with T4 GPU.
______

---
## Table of Contents

- [§0 — Setup](#0--setup)
- [§1 — MLP Reference](#1--mlp-reference)
- [§2 — CNN Experiments](#2--cnn-experiments)
  - [§2.1 Architecture Sweep](#21-architecture-sweep)
  - [§2.2 Architecture Refinement](#22-architecture-refinement)
  - [§2.3 Augmentation](#23-augmentation)
  - [§2.4 Regularisation & Loss](#24-regularisation--loss)
  - [§2.5 Ablations](#25-ablations)
- [§3 — Ensembles](#3--ensembles)
  - [§3.1 Per-class F1 Heatmap](#31-per-class-f1-heatmap)
  - [§3.2 Run Ensemble Combinations](#32-run-ensemble-combinations)
- [§4 — Evaluation & Comparison](#4--evaluation--comparison)
  - [§4.1 Leaderboard](#41-leaderboard)
  - [§4.2 Best Model Deep Dive](#42-best-model-deep-dive)
  - [§4.3 MLP vs CNN Comparison](#43-mlp-vs-cnn-comparison)
- [§5 — Final Summary & Submission](#5--final-summary--submission)

---

**About the code:**

> All reusable logic lives in `src/`. This notebook is the runner + readable story.
> - `src/models/cnn.py` — 6 CNN classes: BaseCNN, DeepCNN, WideCNN, ResidualCNN, MultiScaleCNN (+ SEBlock helper)
> - `src/models/mlp.py` — MLP builder: `MLP(layers, img_size, dropout, use_bn, in_channels)`
> - `src/datasets/dataset.py` — transforms (base, augmented, strong aug), DataLoaders, class weights
> - `src/training/` — shared train loop, early stopping
> - `src/evaluation/` — metrics, plots, ensemble, persistence, submission
>
> **Key helper:** `get_best_models(n)` picks the top-n models from `results_tracker`.
> Each step calls it to automatically start from the current winners — no hard-coded names.


**AI Assistance Disclosure**

> AI tools (Claude, GitHub Copilot) were used for debugging, testing, and formatting.
> All core concepts, architecture decisions, and methodology were conceived by the authors.

---
# §0 — Setup

### Run-level constants

In [ ]:
# All run-level constants live HERE.
# Change FAST_RUN to switch between a quick smoke-test and a real Colab run.

# ==============  FLIP THESE FLAGS  ===============================
FAST_RUN         = True   # True = smoke-test (tiny data, 2 epochs) | False = real Colab run
SAVE_IN_EACH_RUN = False   # True = zip outputs to DRIVE_OUTPUTS_DIR after every experiment
USE_DRIVE        = False    # True = enable save/restore of outputs zip
# =================================================================

TASK_NAME = "task2"

# ==== Drive / backup settings =====================================
DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1u2Xw2-4_L5OhPFjY_OsgNbMayrTI-ROR?usp=sharing"
DATA_FILE_NAME   = "the-pokemon-are-out-there-2026-task-1.zip"
OUTPUTS_FILE_NAME = TASK_NAME + "_outputs.zip"

import sys as _sys
DRIVE_OUTPUTS_DIR = (
    "/content/drive/MyDrive/DL_Proj"
    if "google.colab" in _sys.modules
    else str(__import__("pathlib").Path.home() / "DL_Proj")
)

# ======== Training hyperparams =============================================
EPOCHS      = 2     if FAST_RUN else 30
PATIENCE    = 1     if FAST_RUN else 7
LR          = 1e-3
BATCH_SIZE  = 32    if FAST_RUN else 64
IMG_SIZE    = 64
NUM_WORKERS = 8

N_SAMPLES_PER_CLASS = 6   # 54 total for FAST_RUN smoke-test

print(f"TASK_NAME         : {TASK_NAME}")
print(f"FAST_RUN          : {FAST_RUN}  (N_SAMPLES_PER_CLASS={N_SAMPLES_PER_CLASS if FAST_RUN else 'all'})")
print(f"EPOCHS            : {EPOCHS}  |  PATIENCE : {PATIENCE}  |  LR : {LR}")
print(f"BATCH_SIZE        : {BATCH_SIZE}  |  IMG_SIZE : {IMG_SIZE}x{IMG_SIZE}")
print(f"USE_DRIVE         : {USE_DRIVE}  |  SAVE_IN_EACH_RUN : {SAVE_IN_EACH_RUN}")

### Environment, paths, device

In [ ]:
import sys, os, time, json
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if not IN_COLAB:
  %load_ext autoreload
  %autoreload 2

if IN_COLAB:
    if not os.path.exists("/content/DL_Proj"):
        !git clone https://github.com/fmssilva/DL_Proj.git /content/DL_Proj
    else:
        !git -C /content/DL_Proj pull --ff-only
    os.chdir("/content/DL_Proj/assignment_1")
    %pip install -r requirements.txt -q
else:
    cwd = Path(os.getcwd())
    if cwd.name.startswith("task"):
        os.chdir(cwd.parent)

ROOT = os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import torch
import pandas as pd
import matplotlib.pyplot as plt

from src.config import (
    SEED, CLASSES, NUM_CLASSES, DATA_DIR, OUT_DIR,
    get_task_out_dir, set_seed,
)

set_seed(SEED)

TASK_OUT_DIR = get_task_out_dir(TASK_NAME)
device       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CSV_PATH     = DATA_DIR / "train_labels.csv"
TRAIN_DIR    = DATA_DIR / "Train"
TEST_DIR     = DATA_DIR / "Test"

import multiprocessing
NUM_WORKERS = min(NUM_WORKERS, multiprocessing.cpu_count())

print("-"*50)
print(f"ROOT        : {ROOT}")
print(f"TASK_NAME   : {TASK_NAME}  |  TASK_OUT_DIR : {TASK_OUT_DIR}")
print(f"EPOCHS      : {EPOCHS}  |  PATIENCE : {PATIENCE}  |  LR : {LR}")
print(f"BATCH_SIZE  : {BATCH_SIZE}  |  IMG_SIZE : {IMG_SIZE}x{IMG_SIZE}")
print(f"NUM_WORKERS : {NUM_WORKERS}")
print(f"Device      : {device}  |  PyTorch {torch.__version__}")
print(f"Outputs     : {TASK_OUT_DIR.resolve()}")

### Hot Reload

> If we changed `src/` files and pushed to GitHub, run this cell for Colab to pull them.

In [ ]:
import importlib
import sys
from pathlib import Path

if IN_COLAB:
    import os
    os.chdir(Path(ROOT).parent)
    os.system("git pull --ff-only")
    os.chdir(ROOT)

_reloaded, _skipped = [], []
for mod_name, mod in list(sys.modules.items()):
    if mod_name.startswith("src.") and hasattr(mod, "__file__") and mod.__file__:
        try:
            importlib.reload(mod)
            _reloaded.append(mod_name)
        except Exception as e:
            print(f"  [WARN] could not reload {mod_name}: {e}")
            _skipped.append(mod_name)

print(f"Reloaded {len(_reloaded)} src modules: {_reloaded}")
if _skipped:
    print(f"Skipped  {len(_skipped)}: {_skipped}")

### Data download & subsample

In [ ]:
from src.evaluation.persistence import download_and_extract

if not Path("data/train_labels.csv").exists():
    ok = download_and_extract(DRIVE_FOLDER_URL, DATA_FILE_NAME, "data/")
    if not ok:
        print(
            "ERROR: data download failed.\n"
            f"  Folder : {DRIVE_FOLDER_URL}\n"
            f"  File   : {DATA_FILE_NAME}\n"
            "  Check that the folder is publicly shared and the file name is correct."
        )
else:
    print("data/ already present -- skipping download")

df_full = pd.read_csv(CSV_PATH)
print(f"Full dataset : {len(df_full)} rows, {df_full['label'].nunique()} classes")

if FAST_RUN:
    df = df_full.groupby("label", group_keys=False).head(N_SAMPLES_PER_CLASS).reset_index(drop=True)
    print(f"FAST_RUN     : subsampled to {len(df)} rows ({N_SAMPLES_PER_CLASS}/class)")
else:
    df = df_full

print(f"\nClass counts used this run:\n{df['label'].value_counts().to_string()}")

### Output directories & results restore

### Results tracker & `get_best_models()` helper

In [ ]:
from src.evaluation.persistence import restore_outputs

if USE_DRIVE:
    backup_dir = DRIVE_OUTPUTS_DIR if IN_COLAB else DRIVE_FOLDER_URL
    restore_outputs(TASK_OUT_DIR, TASK_NAME, in_colab=IN_COLAB, backup_dir=backup_dir)
else:
    print("[restore] USE_DRIVE=False -- skipping restore, starting fresh.")

In [ ]:
from src.evaluation.persistence import restore_tracker, ResultsTracker

results_tracker: ResultsTracker = {}
results_json_path = TASK_OUT_DIR / "results" / f"{TASK_NAME}_results.json"


def get_best_models(
    n: int = 1,
    exclude_prefixes: tuple = ("ENS_",),
    exclude_names: tuple = (),
    only_in_registry: bool = False,
) -> list[str]:
    """Return the top-n experiment names sorted by val_macro_f1 (descending).

    - exclude_prefixes: skip names starting with any of these (e.g. 'ENS_')
    - exclude_names: skip these exact names (e.g. 'MLP_ref')
    - only_in_registry: if True, skip names not in model_registry (stale restored entries)
    """
    candidates = {
        k: v for k, v in results_tracker.items()
        if not any(k.startswith(p) for p in exclude_prefixes)
        and k not in exclude_names
        and (not only_in_registry or k in model_registry)
    }
    ranked = sorted(candidates, key=lambda k: candidates[k]["val_macro_f1"], reverse=True)
    return ranked[:n]


def _print_leaderboard(tracker: ResultsTracker) -> None:
    """Show all experiments sorted by val_macro_f1 descending."""
    if not tracker:
        return
    rows = sorted(tracker.items(), key=lambda x: x[1]["val_macro_f1"], reverse=True)
    data = [
        {
            "rank":    rank,
            "name":    name,
            "val_F1":  round(m["val_macro_f1"], 4),
            "val_acc": round(m["val_acc"], 4),
            "epochs":  m["total_epochs"],
            "time(s)": round(m["train_time_s"], 1),
        }
        for rank, (name, m) in enumerate(rows, 1)
    ]
    df_lb = pd.DataFrame(data).set_index("rank")
    try:
        from IPython.display import display
        styled = (
            df_lb.style
            .background_gradient(subset=["val_F1"], cmap="RdYlGn", vmin=0.10, vmax=df_lb["val_F1"].max())
            .background_gradient(subset=["val_acc"], cmap="Blues", vmin=0.10, vmax=df_lb["val_acc"].max())
            .format({"val_F1": "{:.4f}", "val_acc": "{:.4f}", "time(s)": "{:.1f}"})
        )
        display(styled)
    except Exception:
        print(df_lb.to_string())

print("-" * 50)
restore_tracker(results_json_path, results_tracker)
print("Current experiments in results_tracker:")
_print_leaderboard(results_tracker)

### Shared imports, loaders, `run_experiment()`

Defines the core infrastructure used by every experiment cell:
- **class_weights** — inverse-frequency weights for CE loss (handles class imbalance)
- **build_loaders()** — factory for (train, val) DataLoader pairs with augmentation options
- **run_experiment()** — full train loop with early stopping, checkpoint saving, leaderboard update, auto-submission

In [ ]:
# ── Shared imports for all experiment cells ───────────────────────────────────
import torch.nn as nn
from torch.utils.data import DataLoader
from src.datasets.dataset import (
    PokemonDataset, compute_class_weights,
    get_base_transforms, get_augment_transforms, get_strong_aug_transforms,
    get_train_val_loaders,
)
from src.models.cnn import BaseCNN, DeepCNN, WideCNN, ResidualCNN, MultiScaleCNN
from src.models.mlp import MLP  # for MLP reference and cross-task ensemble
from src.training.train import train_one_epoch, evaluate
from src.training.early_stopping import EarlyStopping
from src.evaluation.metrics import classification_report_str
from src.evaluation.plots import plot_history, plot_confusion_matrix
from src.evaluation.persistence import save_experiment_result, save_all_results, save_outputs
from src.evaluation.submission import generate_submission_from_preds, validate_submission

import time

# class weights from the training split
label_to_idx     = {cls: i for i, cls in enumerate(CLASSES)}
_all_train_labels = [label_to_idx[lbl] for lbl in df["label"]]
class_weights    = compute_class_weights(_all_train_labels).to(device)

# test loader — never touched until submission
test_ds     = PokemonDataset(TEST_DIR, get_base_transforms(IMG_SIZE), csv_path=None)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

RESULTS_PATH = TASK_OUT_DIR / "results" / f"{TASK_NAME}_results.json"
SUB_PATH     = TASK_OUT_DIR / "results" / f"submission_{TASK_NAME}.csv"

print(f"Test images  : {len(test_ds)}")
print(f"Class weights: { {cls: round(class_weights[i].item(), 3) for i, cls in enumerate(CLASSES)} }")


# ── build_loaders ─────────────────────────────────────────────────────────────
def build_loaders(augment: bool = False, use_sampler: bool = False,
                  strong_aug: bool = False):
    """Return (train_loader, val_loader). strong_aug overrides augment."""
    train_t = (
        get_strong_aug_transforms(IMG_SIZE) if strong_aug
        else get_augment_transforms(IMG_SIZE) if augment
        else None
    )
    return get_train_val_loaders(
        CSV_PATH, TRAIN_DIR, IMG_SIZE, BATCH_SIZE,
        augment=False,
        use_sampler=use_sampler,
        num_workers=NUM_WORKERS, df_override=df,
        train_transform=train_t,
    )


# pre-build the two standard loader pairs
loaders_base    = build_loaders(augment=False, use_sampler=False)
loaders_sampler = build_loaders(augment=False, use_sampler=True)
loaders_aug     = build_loaders(augment=True,  use_sampler=False)
loaders_strong  = build_loaders(strong_aug=True, use_sampler=False)


# ── _is_new_overall_best ──────────────────────────────────────────────────────
def _is_new_overall_best(name: str, f1: float) -> bool:
    """True if this experiment beats every other entry in results_tracker."""
    other_f1s = [v["val_macro_f1"] for k, v in results_tracker.items() if k != name]
    return f1 > max(other_f1s, default=-1)


# ── run_experiment ────────────────────────────────────────────────────────────
def run_experiment(model, name: str, criterion, optimizer, scheduler, loaders) -> tuple:
    """
    Train model up to EPOCHS with early stopping on val macro-F1.
    Saves best checkpoint, upserts results JSON, auto-updates submission if new best.
    Returns (model_with_best_weights, history_dict).
    """
    train_loader, val_loader = loaders
    ckpt_path = str(TASK_OUT_DIR / "checkpoints" / f"{name}.pth")
    stopper   = EarlyStopping(patience=PATIENCE, checkpoint_path=ckpt_path)
    history   = {"train_loss": [], "val_loss": [], "train_f1": [], "val_f1": []}
    t0        = time.time()

    print(f"\n{'='*60}")
    print(f"  Experiment: {name}")
    print(f"  Model params: {sum(p.numel() for p in model.parameters()):,}")
    print(f"{'='*60}")

    for epoch in range(1, EPOCHS + 1):
        ep_t0         = time.time()
        train_loss    = train_one_epoch(model, train_loader, criterion, optimizer, device)
        train_metrics = evaluate(model, train_loader, criterion, device)
        val_metrics   = evaluate(model, val_loader,   criterion, device)
        elapsed       = time.time() - ep_t0
        scheduler.step()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_metrics["loss"])
        history["train_f1"].append(train_metrics["macro_f1"])
        history["val_f1"].append(val_metrics["macro_f1"])

        print(
            f"  Epoch {epoch:02d}/{EPOCHS} | "
            f"train_loss={train_loss:.4f}  train_f1={train_metrics['macro_f1']:.4f} | "
            f"val_loss={val_metrics['loss']:.4f}  val_f1={val_metrics['macro_f1']:.4f} | "
            f"{elapsed:.1f}s"
        )

        stopper(-val_metrics["macro_f1"], model)
        if stopper.stop:
            print(f"  Early stopping at epoch {epoch} (patience={PATIENCE}, metric: val_macro_f1).")
            break

    total_time = time.time() - t0

    # reload best checkpoint
    model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
    best = evaluate(model, val_loader, criterion, device)

    entry = {
        "val_macro_f1": round(best["macro_f1"], 4),
        "val_acc":      round(best["acc"], 4),
        "val_loss":     round(best["loss"], 4),
        "total_epochs": len(history["train_loss"]),
        "train_time_s": round(total_time, 1),
        "history":      history,
    }
    results_tracker[name] = entry

    print(f"\n  Best checkpoint: val_loss={best['loss']:.4f}  val_macro_f1={best['macro_f1']:.4f}")
    print(f"  Total time: {total_time:.1f}s ({total_time/len(history['train_loss']):.1f}s/epoch)")
    _print_leaderboard(results_tracker)

    save_experiment_result(name, entry, RESULTS_PATH)

    # auto-update submission if this is the new overall best
    if _is_new_overall_best(name, entry["val_macro_f1"]):
        print(f"\n  New overall best -- regenerating submission CSV...")
        test_preds = []
        model.eval()
        with torch.no_grad():
            for imgs, _ in test_loader:
                imgs = imgs.to(device)
                test_preds.extend(model(imgs).argmax(dim=1).cpu().tolist())
        generate_submission_from_preds(test_loader, test_preds, CLASSES, SUB_PATH)
        validate_submission(SUB_PATH)
        print(f"  Submission updated -> {SUB_PATH}")

    if SAVE_IN_EACH_RUN:
        save_outputs(TASK_OUT_DIR, TASK_NAME, in_colab=IN_COLAB, use_drive=USE_DRIVE, drive_dir=DRIVE_OUTPUTS_DIR)

    return model, history

### Model Registry

Maps experiment name → `{model: factory, criterion: factory}`.
Used by the heatmap, ensemble, and analysis cells to reconstruct any model from its checkpoint.

Only §2.1 entries are registered here (static, known architectures).
Later steps add entries dynamically after their experiments run.

In [ ]:
model_registry = {
    # ── MLP reference (Task 1 best recipe) ─────────────────────────────────
    "MLP_ref": {
        "model": lambda: MLP(layers=[512, 256, 128], img_size=IMG_SIZE, dropout=0.3),
        "criterion": lambda: nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.15),
    },
    # ── Phase 1.1: Architecture sweep — each CNN with default recipe ───────
    "A_base_cnn": {
        "model": lambda: BaseCNN(dropout=0.5),
        "criterion": lambda: nn.CrossEntropyLoss(weight=class_weights),
    },
    "B_wide": {
        "model": lambda: WideCNN(dropout=0.5),
        "criterion": lambda: nn.CrossEntropyLoss(weight=class_weights),
    },
    "C_deep": {
        "model": lambda: DeepCNN(dropout=0.5),
        "criterion": lambda: nn.CrossEntropyLoss(weight=class_weights),
    },
    "D_residual": {
        "model": lambda: ResidualCNN(dropout=0.5, use_se=False),
        "criterion": lambda: nn.CrossEntropyLoss(weight=class_weights),
    },
    "E_residual_se": {
        "model": lambda: ResidualCNN(dropout=0.5, use_se=True),
        "criterion": lambda: nn.CrossEntropyLoss(weight=class_weights),
    },
    "F_multiscale": {
        "model": lambda: MultiScaleCNN(dropout=0.5),
        "criterion": lambda: nn.CrossEntropyLoss(weight=class_weights),
    },
    # later phases register their entries dynamically via checkpoint cells
}
print(f"Model registry: {len(model_registry)} entries")
print("  " + ", ".join(model_registry.keys()))

---
# §1 — MLP Reference

Re-run the best MLP recipe from Task 1 (`MLP([512,256,128], drop=0.3, LS=0.15, class_weights, StepLR`)).
This gives us an in-notebook baseline number so every CNN can be compared directly.

In [ ]:
# ── MLP Reference: best Task 1 recipe ────────────────────────────────────────
# MLP([512, 256, 128], drop=0.3) + CE(weights, LS=0.15) + StepLR
# Expected: val_F1 ~ 0.24 (same as Task 1 R_ls015_drop03)

model_mlp = MLP(layers=[512, 256, 128], img_size=IMG_SIZE, dropout=0.3).to(device)
criterion_mlp = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.15)
optimizer_mlp = torch.optim.Adam(model_mlp.parameters(), lr=LR)
scheduler_mlp = torch.optim.lr_scheduler.StepLR(optimizer_mlp, step_size=5, gamma=0.5)

model_mlp, history_mlp = run_experiment(
    model_mlp, "MLP_ref", criterion_mlp, optimizer_mlp, scheduler_mlp, loaders_base
)


---
# §2 — CNN Experiments

Progressive refinement — change **one variable at a time** so we can isolate each effect:

| Step | Variable changed | Held constant |
|------|-----------------|---------------|
| §2.1 | Architecture (6 CNNs) | No aug, same optimizer/loss |
| §2.2 | Architecture tweaks | Same as §2.1 |
| §2.3 | Augmentation pipeline | Best architecture(s) from §2.1–2.2 |
| §2.4 | Loss / sampling strategy | Best architecture + best augmentation |
| §2.5 | Ablations (remove components) | Best full model from above |

## §2.1 — Architecture Sweep (no augmentation)

Clean comparison — **same recipe for all**, only the architecture changes.
Recipe: `CE(class_weights) + Dropout(0.5) + Adam(1e-3) + StepLR(5, 0.5)`, no augmentation.

| ID | Architecture | ~Params | What it tests |
|----|-------------|---------|---------------|
| A | BaseCNN (3 blocks, 32→64→128) | 95K | CNN baseline — beat MLP? |
| B | WideCNN (3 blocks, 64→128→256) | 374K | Width: richer per-layer features |
| C | DeepCNN (4 blocks, 32→…→256) | 392K | Depth: spatial down to 4×4 |
| D | ResidualCNN (3 res-blocks, 3→64→128→256) | 1.2M | Skip connections |
| E | ResidualCNN+SE (+ channel attention) | 1.2M | Squeeze-and-Excitation attention |
| F | MultiScaleCNN (k=3 ∥ k=5 parallel) | 178K | Multi-scale texture capture |

#### A — BaseCNN (~95K params)
`Conv32→Pool→Conv64→Pool→Conv128→Pool→GAP→Drop(0.5)→Linear(9)`

In [ ]:
model_A = BaseCNN(dropout=0.5).to(device)
criterion_A = nn.CrossEntropyLoss(weight=class_weights)
optimizer_A = torch.optim.Adam(model_A.parameters(), lr=LR)
scheduler_A = torch.optim.lr_scheduler.StepLR(optimizer_A, step_size=5, gamma=0.5)

model_A, history_A = run_experiment(model_A, "A_base_cnn", criterion_A, optimizer_A, scheduler_A, loaders_base)

#### B — WideCNN (~374K params)
`Conv64→Pool→Conv128→Pool→Conv256→Pool→GAP→Drop(0.5)→Linear(9)`

In [ ]:
model_B = WideCNN(dropout=0.5).to(device)
criterion_B = nn.CrossEntropyLoss(weight=class_weights)
optimizer_B = torch.optim.Adam(model_B.parameters(), lr=LR)
scheduler_B = torch.optim.lr_scheduler.StepLR(optimizer_B, step_size=5, gamma=0.5)

model_B, history_B = run_experiment(model_B, "B_wide", criterion_B, optimizer_B, scheduler_B, loaders_base)

#### C — DeepCNN (~392K params)
`Conv32→Conv64→Conv128→Conv256→GAP→Drop(0.5)→Linear(9)` — 4 blocks, spatial down to 4×4

In [ ]:
model_C = DeepCNN(dropout=0.5).to(device)
criterion_C = nn.CrossEntropyLoss(weight=class_weights)
optimizer_C = torch.optim.Adam(model_C.parameters(), lr=LR)
scheduler_C = torch.optim.lr_scheduler.StepLR(optimizer_C, step_size=5, gamma=0.5)

model_C, history_C = run_experiment(model_C, "C_deep", criterion_C, optimizer_C, scheduler_C, loaders_base)

#### D — ResidualCNN (~1.2M params)
`ResBlock(3→64)→ResBlock(64→128)→ResBlock(128→256)→GAP→Drop(0.5)→Linear(9)`
Each block: `Conv→BN→ReLU→Conv→BN→(+skip)→ReLU→Pool`

In [ ]:
model_D = ResidualCNN(dropout=0.5, use_se=False).to(device)
criterion_D = nn.CrossEntropyLoss(weight=class_weights)
optimizer_D = torch.optim.Adam(model_D.parameters(), lr=LR)
scheduler_D = torch.optim.lr_scheduler.StepLR(optimizer_D, step_size=5, gamma=0.5)

model_D, history_D = run_experiment(model_D, "D_residual", criterion_D, optimizer_D, scheduler_D, loaders_base)

#### E — ResidualCNN + SE (~1.2M params)
Same as D but with Squeeze-and-Excitation channel attention after each residual block.

In [ ]:
model_E = ResidualCNN(dropout=0.5, use_se=True).to(device)
criterion_E = nn.CrossEntropyLoss(weight=class_weights)
optimizer_E = torch.optim.Adam(model_E.parameters(), lr=LR)
scheduler_E = torch.optim.lr_scheduler.StepLR(optimizer_E, step_size=5, gamma=0.5)

model_E, history_E = run_experiment(model_E, "E_residual_se", criterion_E, optimizer_E, scheduler_E, loaders_base)

#### F — MultiScaleCNN (~178K params)
`[Conv(k=3) ∥ Conv(k=5)]→cat→BN→ReLU→Pool` × 3 blocks → `GAP→Drop(0.5)→Linear(9)`

In [ ]:
model_F = MultiScaleCNN(dropout=0.5).to(device)
criterion_F = nn.CrossEntropyLoss(weight=class_weights)
optimizer_F = torch.optim.Adam(model_F.parameters(), lr=LR)
scheduler_F = torch.optim.lr_scheduler.StepLR(optimizer_F, step_size=5, gamma=0.5)

model_F, history_F = run_experiment(model_F, "F_multiscale", criterion_F, optimizer_F, scheduler_F, loaders_base)

## §2.2 — Architecture Refinement

After reviewing §2.1 results, try small tweaks on the top architectures
(e.g. different channel widths, extra blocks, combining ideas from different models).
Copy the template below for each variant.

In [ ]:
# ── Phase 1.2 template — copy this cell for each architecture variant ─────────
# Example: ResidualCNN with SE + wider channels, or DeepCNN with more blocks, etc.

# 1. register in model_registry so heatmap/ensemble can reconstruct it later
# model_registry["X_variant_name"] = {
#     "model":     lambda: ResidualCNN(dropout=0.5, use_se=True),
#     "criterion": lambda: nn.CrossEntropyLoss(weight=class_weights),
# }

# 2. run the experiment
# model_X = model_registry["X_variant_name"]["model"]().to(device)
# criterion_X = model_registry["X_variant_name"]["criterion"]()
# optimizer_X = torch.optim.Adam(model_X.parameters(), lr=LR)
# scheduler_X = torch.optim.lr_scheduler.StepLR(optimizer_X, step_size=5, gamma=0.5)
# model_X, history_X = run_experiment(model_X, "X_variant_name", criterion_X, optimizer_X, scheduler_X, loaders_base)

print("Phase 1.2: no variants added yet — fill this in after reviewing Phase 1.1 results.")

### §2.1–2.2 Checkpoint — pick winners for §2.3

In [ ]:
# pick top-2 CNN architectures from Phase 1 (exclude MLP_ref and ensembles)
# only_in_registry=True ensures we don't pick stale entries from a restored JSON
phase1_winners = get_best_models(n=2, exclude_names=("MLP_ref",), only_in_registry=True)
mlp_ref_f1 = results_tracker.get("MLP_ref", {}).get("val_macro_f1", 0)

print("Phase 1 winners:")
for i, name in enumerate(phase1_winners, 1):
    f1 = results_tracker[name]["val_macro_f1"]
    print(f"  #{i}: {name}  (val_F1={f1:.4f}, vs MLP: {f1 - mlp_ref_f1:+.4f})")

print(f"\nMLP reference: MLP_ref  (val_F1={mlp_ref_f1:.4f})")
_print_leaderboard(results_tracker)

---
## §2.3 — Augmentation on Best Architectures

**Why augmentation before regularisation?**
Augmentation changes the effective training distribution — it's a more fundamental change than
tweaking dropout or label smoothing. The optimal regularisation often depends on the data pipeline,
so we lock down the best augmentation first, then tune regularisation on top of it.

**Approach:** Grid search — top-2 architectures × 2 augmentation pipelines:
- **aug** — standard: HorizontalFlip + ColorJitter + Rotation(15°)
- **saug** — strong: + VerticalFlip + Affine + RandomErasing

This gives us `2 × 2 = 4` runs — affordable on our 1h budget.

In [ ]:
# ── Phase 2: augmentation grid ────────────────────────────────────────────────
# Run top Phase 1 architectures through different augmentation pipelines.
# Each combination gets a name like "aug_D_residual" or "saug_E_residual_se".

# define the pipelines to test (name_prefix -> loaders)
PIPELINES = {
    "aug":  loaders_aug,       # standard augmentation
    "saug": loaders_strong,    # strong augmentation
}

# take the Phase 1 winners (run the checkpoint cell above first)
BASE_MODELS = phase1_winners   # e.g. ["D_residual", "E_residual_se"]

for base_name in BASE_MODELS:
    base_entry = model_registry[base_name]
    for pipe_prefix, pipe_loaders in PIPELINES.items():
        exp_name = f"{pipe_prefix}_{base_name}"
        print(f"\n--- {exp_name} ---")

        model = base_entry["model"]().to(device)
        criterion = base_entry["criterion"]()
        optimizer = torch.optim.Adam(model.parameters(), lr=LR)
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

        model, history = run_experiment(model, exp_name, criterion, optimizer, scheduler, pipe_loaders)

        # register so heatmap/ensemble can reconstruct it — same architecture, just different training data
        model_registry[exp_name] = {
            "model":     base_entry["model"],
            "criterion": base_entry["criterion"],
        }

---
## §2.4 — Regularisation & Loss Variants

Start from the current best model (could be §2.1, §2.2, or §2.3 winner).
Try loss/sampling strategies that Task 1 showed were impactful:
- **sampler** — WeightedRandomSampler (no class weights in loss — combining both hurts)
- **ls** — Label smoothing = 0.1 (confirmed +0.03 for MLP in Task 1)

In [ ]:
# ── Phase 3 checkpoint: pick the current best (any phase) ─────────────────────
# only_in_registry=True ensures we pick a model we can actually reconstruct
phase3_base = get_best_models(n=1, exclude_names=("MLP_ref",), only_in_registry=True)[0]
phase3_entry = model_registry[phase3_base]
print(f"Phase 3 base model: {phase3_base}  (val_F1={results_tracker[phase3_base]['val_macro_f1']:.4f})")

# figure out the best augmentation pipeline for this model
# if the winner is from Phase 2, reuse its pipeline; otherwise use standard aug
phase3_strong_aug = phase3_base.startswith("saug_")
if phase3_strong_aug:
    phase3_loaders = loaders_strong
elif phase3_base.startswith("aug_"):
    phase3_loaders = loaders_aug
else:
    phase3_loaders = loaders_aug   # default to standard aug for Phase 1 winners

print(f"  Pipeline: {'strong_aug' if phase3_strong_aug else 'aug'}")

In [ ]:
# ── Phase 3 experiments ───────────────────────────────────────────────────────
# Each variant uses the same architecture as phase3_base, just a different loss or sampling strategy.
# The augmentation pipeline stays the same as the Phase 3 base — we only vary the regularisation.

REG_VARIANTS = {
    # sampler: rebalance batches instead of weighting the loss
    f"reg_{phase3_base}_sampler": {
        "criterion": lambda: nn.CrossEntropyLoss(),  # no weights — sampler handles imbalance
        "loaders": build_loaders(strong_aug=phase3_strong_aug, augment=(not phase3_strong_aug), use_sampler=True),
    },
    # label smoothing: softens one-hot targets, helps calibration
    f"reg_{phase3_base}_ls01": {
        "criterion": lambda: nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1),
        "loaders": phase3_loaders,
    },
}

for exp_name, variant in REG_VARIANTS.items():
    print(f"\n--- {exp_name} ---")
    model = phase3_entry["model"]().to(device)
    criterion = variant["criterion"]()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)
    model, history = run_experiment(model, exp_name, criterion, optimizer, scheduler, variant["loaders"])

    # register — same model factory, different criterion
    model_registry[exp_name] = {
        "model": phase3_entry["model"],
        "criterion": variant["criterion"],
    }

---
## §2.5 — Ablations

Confirm that each component of the best model actually earns its place.
We take the **overall best CNN so far** and remove one thing at a time to measure its impact.

In [ ]:
# ── §2.5 checkpoint: pick the overall best CNN for ablation ────────────────────
ablation_base = get_best_models(n=1, exclude_names=("MLP_ref",), only_in_registry=True)[0]
ablation_entry = model_registry[ablation_base]
ablation_f1 = results_tracker[ablation_base]["val_macro_f1"]
print(f"Ablation base: {ablation_base}  (val_F1={ablation_f1:.4f})")

# determine augmentation pipeline used by this model (for consistent comparison)
ablation_strong_aug = ablation_base.startswith("saug_")
if ablation_strong_aug:
    ablation_loaders = loaders_strong
elif ablation_base.startswith("aug_") or ablation_base.startswith("reg_"):
    ablation_loaders = loaders_aug
else:
    ablation_loaders = loaders_base  # Phase 1 winners used no augmentation

print(f"  Pipeline: {'strong_aug' if ablation_strong_aug else 'aug' if ablation_loaders is loaders_aug else 'base'}")
print(f"\nAblation plan: re-train the same architecture, removing one component each time.")
print(f"  - Lower dropout (0.3 vs 0.5)")
print(f"  - No BatchNorm")
print(f"  - CosineAnnealingLR vs StepLR")

In [ ]:
# ── §2.5 ablation experiments ──────────────────────────────────────────────────
# Each variant removes or changes one component of the best model.
# We build ablation variants dynamically from the best model's class and settings.

import inspect

# figure out the model class and its constructor kwargs
_ref_model = ablation_entry["model"]()
_model_cls = type(_ref_model)
_cls_params = inspect.signature(_model_cls.__init__).parameters
del _ref_model

# default kwargs that reconstructed the best model (same as registry)
_default_kwargs = {}
if "dropout" in _cls_params:    _default_kwargs["dropout"] = 0.5
if "use_se" in _cls_params:     _default_kwargs["use_se"] = True if "se" in ablation_base.lower() else False
if "use_bn" in _cls_params:     _default_kwargs["use_bn"] = True

print(f"Model class: {_model_cls.__name__}, kwargs: {_default_kwargs}")

# -- define ablation variants -------------------------------------------------
ABLATIONS = {}

# 1. lower dropout (0.3 vs 0.5) — is our default too aggressive?
if "dropout" in _cls_params:
    lo_drop_kw = {**_default_kwargs, "dropout": 0.3}
    ABLATIONS[f"abl_{ablation_base}_drop03"] = {
        "model":     lambda kw=lo_drop_kw: _model_cls(**kw),
        "criterion": ablation_entry["criterion"],
        "loaders":   ablation_loaders,
        "scheduler_fn": lambda opt: torch.optim.lr_scheduler.StepLR(opt, step_size=5, gamma=0.5),
        "why": "Dropout 0.3 vs 0.5 — is default too aggressive?",
    }

# 2. no BatchNorm — only if the architecture supports use_bn
if "use_bn" in _cls_params:
    no_bn_kw = {**_default_kwargs, "use_bn": False}
    ABLATIONS[f"abl_{ablation_base}_no_bn"] = {
        "model":     lambda kw=no_bn_kw: _model_cls(**kw),
        "criterion": ablation_entry["criterion"],
        "loaders":   ablation_loaders,
        "scheduler_fn": lambda opt: torch.optim.lr_scheduler.StepLR(opt, step_size=5, gamma=0.5),
        "why": "Remove BatchNorm — quantify BN's contribution",
    }

# 3. CosineAnnealingLR instead of StepLR — smoother decay
ABLATIONS[f"abl_{ablation_base}_cosine_lr"] = {
    "model":     ablation_entry["model"],
    "criterion": ablation_entry["criterion"],
    "loaders":   ablation_loaders,
    "scheduler_fn": lambda opt: torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-5),
    "why": "CosineAnnealingLR vs StepLR — smoother schedule",
}

# -- run all ablations --------------------------------------------------------
print(f"\nRunning {len(ABLATIONS)} ablation(s):")
for exp_name, abl in ABLATIONS.items():
    print(f"\n--- {exp_name}: {abl['why']} ---")
    model = abl["model"]().to(device)
    criterion = abl["criterion"]()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = abl["scheduler_fn"](optimizer)
    model, history = run_experiment(model, exp_name, criterion, optimizer, scheduler, abl["loaders"])

    # register so heatmap/ensemble can reconstruct this model later
    model_registry[exp_name] = {
        "model":     abl["model"],
        "criterion": abl["criterion"],
    }

# show impact summary
print("\n" + "=" * 60)
print(f"  Ablation summary (baseline: {ablation_base} = {ablation_f1:.4f})")
print("=" * 60)
for exp_name, abl in ABLATIONS.items():
    abl_f1 = results_tracker.get(exp_name, {}).get("val_macro_f1", 0)
    delta = abl_f1 - ablation_f1
    print(f"  {exp_name:40s}  val_F1={abl_f1:.4f}  ({delta:+.4f})  | {abl['why']}")
_print_leaderboard(results_tracker)

---
# §3 — Ensembles

Combine complementary models to squeeze extra F1 via soft voting (average softmax probabilities).
The per-class heatmap below helps pick models that are strong on **different** classes —
diversity beats having 3 copies of the same architecture.

In [ ]:
from src.evaluation.ensemble import soft_ensemble, print_ensemble_report

def run_ensemble(names: list, weights: list = None, label: str = None, print_leaderboard: bool = False):
    """Run a soft-vote ensemble. Works with ANY model type in model_registry.

    Args:
        names:   list of checkpoint stems (must be in model_registry)
        weights: optional per-model weights (None = uniform). Auto-normalised.
        label:   ensemble name for results_tracker. Default: "ENS_<name1>_<name2>_..."
        print_leaderboard: show full leaderboard after run
    """
    if label is None:
        label = "ENS_" + "_".join(names)

    # validate all members exist in registry + have checkpoints
    for n in names:
        if n not in model_registry:
            print(f"SKIP {label}: '{n}' not in model_registry")
            return
    ckpt_paths = [TASK_OUT_DIR / "checkpoints" / f"{n}.pth" for n in names]
    missing = [str(p) for p in ckpt_paths if not p.exists()]
    if missing:
        print(f"SKIP {label}: missing checkpoints: {missing}")
        return

    # build (model_instance, ckpt_path) pairs — architecture from registry guarantees correct match
    configs = [(model_registry[n]["model"](), str(p)) for n, p in zip(names, ckpt_paths)]

    # evaluate on validation set (always base transforms for fair comparison)
    _, val_loader_ens = build_loaders()
    ens_result = soft_ensemble(configs, val_loader_ens, device, CLASSES, weights=weights)
    print_ensemble_report(ens_result, ensemble_label=label)

    # save to results_tracker
    entry = {
        "val_macro_f1": ens_result["val_macro_f1"],
        "val_acc":      ens_result["val_acc"],
        "val_loss":     float("nan"),
        "total_epochs": 0,
        "train_time_s": 0.0,
        "history":      {"train_loss": [], "val_loss": [], "train_f1": [], "val_f1": []},
        "child_models": names,
    }
    results_tracker[label] = entry
    save_experiment_result(label, entry, RESULTS_PATH)

    # auto-update submission if this ensemble is the new overall best
    if _is_new_overall_best(label, entry["val_macro_f1"]):
        print(f"\n  New overall best — regenerating submission CSV...")
        test_configs = [(model_registry[n]["model"](), str(TASK_OUT_DIR / "checkpoints" / f"{n}.pth")) for n in names]
        test_result = soft_ensemble(test_configs, test_loader, device, CLASSES, weights=weights, inference_mode=True)
        generate_submission_from_preds(test_loader, test_result["preds"], CLASSES, SUB_PATH)
        validate_submission(SUB_PATH)
        print(f"  Submission updated -> {SUB_PATH}")

    if SAVE_IN_EACH_RUN:
        save_outputs(TASK_OUT_DIR, TASK_NAME, in_colab=IN_COLAB, use_drive=USE_DRIVE, drive_dir=DRIVE_OUTPUTS_DIR)

    if print_leaderboard:
        _print_leaderboard(results_tracker)

    return entry

## §3.1 — Per-class F1 Heatmap

Which models are strong/weak on each class?
Good ensemble candidates have **complementary** weaknesses — if model A fails on Rock
and model B fails on Water, combining them covers both gaps.

In [ ]:
from src.evaluation.analysis import plot_per_class_f1_heatmap

# the heatmap loads each checkpoint, builds the model from model_registry, and evaluates per-class F1
fig = plot_per_class_f1_heatmap(
    checkpoint_dir=TASK_OUT_DIR / "checkpoints",
    model_registry={k: v["model"] for k, v in model_registry.items()},
    loader_fn=build_loaders,
    device=device,
    out_path=TASK_OUT_DIR / "plots" / f"{TASK_NAME}_per_class_f1_heatmap.png",
    highlight_best=True,
)
if fig:
    plt.show()
    plt.close(fig)

## §3.2 — Run Ensemble Combinations

Copy-paste the cell below for each combination you want to try.
Look at the heatmap above and pick models with complementary per-class strengths.

In [ ]:
# ── Auto ensembles: top-2 CNNs, top CNN + MLP ────────────────────────────────
top2 = get_best_models(n=2, exclude_names=("MLP_ref",), only_in_registry=True)
print(f"Auto-selected top-2 CNNs: {top2}")

run_ensemble(top2)                         # top-2 CNN solos
run_ensemble([top2[0], "MLP_ref"])         # best CNN + MLP (architectural diversity)

# ── Manual ensembles: copy-paste this block for each combo you want to try ────
run_ensemble(["D_residual", "F_multiscale"])                    # specific pair
# run_ensemble(["D_residual", "E_residual_se", "F_multiscale"])   # 3-way
# run_ensemble(["D_residual", "MLP_ref"], weights=[0.7, 0.3])    # weighted

print("\n" + "=" * 60)
_print_leaderboard(results_tracker)

---
# §4 — Evaluation & Comparison

Deep dive on the final results: leaderboard, best model analysis, and MLP vs CNN comparison.

## §4.1 — Leaderboard

In [ ]:
from src.evaluation.analysis import plot_leaderboard

print("=== All Experiments -- Sorted by Val Macro-F1 ===")
_print_leaderboard(results_tracker)

fig = plot_leaderboard(
    results_tracker,
    out_path=TASK_OUT_DIR / "plots" / f"{TASK_NAME}_leaderboard.png",
)
plt.show()
plt.close(fig)

## §4.2 — Best Model Deep Dive

Reload the overall best solo CNN checkpoint, run classification report,
plot training curves + confusion matrix.
Works for any model type (base, augmented, regularised, or ablation).

In [ ]:
# best solo CNN — exclude ensembles, MLP_ref, and stale entries without a model factory
best_cnn_name = get_best_models(n=1, exclude_names=("MLP_ref",), only_in_registry=True)[0]
print(f"Best solo CNN: {best_cnn_name} (val_F1={results_tracker[best_cnn_name]['val_macro_f1']:.4f})")

# load checkpoint and evaluate on validation set
best_model = model_registry[best_cnn_name]["model"]().to(device)
ckpt = str(TASK_OUT_DIR / "checkpoints" / f"{best_cnn_name}.pth")
best_model.load_state_dict(torch.load(ckpt, map_location=device, weights_only=True))

_, val_loader_eval = build_loaders()
best_model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in val_loader_eval:
        preds = best_model(imgs.to(device)).argmax(dim=1).cpu().tolist()
        all_preds.extend(preds)
        all_labels.extend(labels.tolist())

# per-class classification report
print(f"\n{classification_report_str(all_labels, all_preds, CLASSES)}")

# training curves
history_best = results_tracker[best_cnn_name]["history"]
fig_hist = plot_history(history_best, out_path=TASK_OUT_DIR / "plots" / f"{best_cnn_name}_history.png")
plt.show()
plt.close(fig_hist)

# confusion matrix
fig_cm = plot_confusion_matrix(all_labels, all_preds, CLASSES,
                                out_path=TASK_OUT_DIR / "plots" / f"{best_cnn_name}_confusion.png")
plt.show()
plt.close(fig_cm)

## §4.3 — MLP vs CNN Comparison

Side-by-side: best MLP from Task 1 vs best CNN from this notebook.
Shows whether the inductive biases of convolutions actually help on this dataset.

In [ ]:
# load task1 results for comparison
import json

task1_json = Path("task1/outputs/results/task1_results.json")
if task1_json.exists():
    with open(task1_json) as f:
        task1_data = json.load(f)
    task1_exps = task1_data.get("experiments", {})
    # best solo MLP from task1 (exclude ensembles)
    task1_solo = {k: v for k, v in task1_exps.items() if not k.startswith("ENS_")}
    task1_best_name = max(task1_solo, key=lambda k: task1_solo[k]["val_macro_f1"])
    task1_best_f1 = task1_solo[task1_best_name]["val_macro_f1"]
else:
    task1_best_name = "MLP_ref (in-notebook)"
    task1_best_f1 = results_tracker.get("MLP_ref", {}).get("val_macro_f1", 0)

# best solo CNN
task2_best_f1 = results_tracker[best_cnn_name]["val_macro_f1"]

# best ensemble
ens_names = [k for k in results_tracker if k.startswith("ENS_")]
best_ens_name = max(ens_names, key=lambda k: results_tracker[k]["val_macro_f1"]) if ens_names else None
best_ens_f1 = results_tracker[best_ens_name]["val_macro_f1"] if best_ens_name else 0

print("=" * 60)
print("  MLP vs CNN Comparison")
print("=" * 60)
print(f"  Task 1 best MLP (solo) : {task1_best_name:30s}  val_F1={task1_best_f1:.4f}")
print(f"  Task 2 MLP reference   : {'MLP_ref':30s}  val_F1={results_tracker.get('MLP_ref', {}).get('val_macro_f1', 0):.4f}")
print(f"  Task 2 best CNN (solo) : {best_cnn_name:30s}  val_F1={task2_best_f1:.4f}")
if best_ens_name:
    print(f"  Task 2 best ensemble   : {best_ens_name:30s}  val_F1={best_ens_f1:.4f}")
print(f"\n  CNN vs MLP improvement : {task2_best_f1 - task1_best_f1:+.4f} F1")

# training curves comparison: MLP vs best CNN
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# loss comparison
h_mlp = results_tracker["MLP_ref"]["history"]
h_cnn = results_tracker[best_cnn_name]["history"]
ax1.plot(h_mlp["train_loss"], "--", label="MLP train", color="orange")
ax1.plot(h_mlp["val_loss"], "-", label="MLP val", color="orange")
ax1.plot(h_cnn["train_loss"], "--", label=f"{best_cnn_name} train", color="blue")
ax1.plot(h_cnn["val_loss"], "-", label=f"{best_cnn_name} val", color="blue")
ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss"); ax1.set_title("Loss: MLP vs CNN")
ax1.legend(); ax1.grid(True)

# F1 comparison
ax2.plot(h_mlp["val_f1"], "-", label="MLP val F1", color="orange")
ax2.plot(h_cnn["val_f1"], "-", label=f"{best_cnn_name} val F1", color="blue")
ax2.set_xlabel("Epoch"); ax2.set_ylabel("Macro F1"); ax2.set_title("Val F1: MLP vs CNN")
ax2.legend(); ax2.grid(True)

fig.tight_layout()
fig.savefig(TASK_OUT_DIR / "plots" / "mlp_vs_cnn_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

---
# §5 — Final Summary & Submission

Leaderboard, time budget check, and submission file validation.

In [ ]:
# save all results to the final JSON
print("=" * 60)
print("Final leaderboard:")
_print_leaderboard(results_tracker)

# total training time
total_time = sum(v["train_time_s"] for v in results_tracker.values())
print(f"\nTotal training time: {total_time:.0f}s ({total_time/60:.1f} min)")
print(f"Budget: 3600s (60 min). Used: {total_time/3600*100:.1f}%")

# validate final submission
if SUB_PATH.exists():
    validate_submission(SUB_PATH)
    print(f"\nSubmission file: {SUB_PATH}")
else:
    print("\nWARNING: No submission file found.")

In [ ]:
# backup outputs to Drive
save_outputs(TASK_OUT_DIR, TASK_NAME, in_colab=IN_COLAB, use_drive=USE_DRIVE, drive_dir=DRIVE_OUTPUTS_DIR)